### Jarvis data extraction 

In [ ]:
import requests
import pandas as pd
import json
import re
import numpy as np 

def lattice_vectors_to_parameters(lattice_vectors):
    """
    Calculation of the lattice parameters (a, b, c, alpha, beta, gamma) out of three lattice vectors.

    Args:
        lattice_vectors: list of three vectors [[ax, ay, az], [bx, by, bz], [cx, cy, cz]]

    Returns:
        dict with a, b, c, alpha, beta, gamma
    """
    a_vec = np.array(lattice_vectors[0])
    b_vec = np.array(lattice_vectors[1])
    c_vec = np.array(lattice_vectors[2])

    # length
    a = np.linalg.norm(a_vec)
    b = np.linalg.norm(b_vec)
    c = np.linalg.norm(c_vec)

    # degree 
    alpha = np.degrees(np.arccos(np.dot(b_vec, c_vec) / (b * c)))
    beta  = np.degrees(np.arccos(np.dot(a_vec, c_vec) / (a * c)))
    gamma = np.degrees(np.arccos(np.dot(a_vec, b_vec) / (a * b)))

    return {
        "a": a,
        "b": b,
        "c": c,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma
    }

def jarvis_optimade_query(query):
    base_url = "https://jarvis.nist.gov/optimade/jarvisdft/v1/structures/?filter="
    url = base_url + query
    results = []

    while url:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        results.extend(data["data"])
        url = data.get("links", {}).get("next")

    return results


# Filter for elements (Refractory Elements) 
refractory_elements = ["W", "Mo", "Ta", "Nb", "V", "Cr", "Hf", "Zr", "Ti", "Re"]
#refractory_elements = ["W"]
entries = []

for el in refractory_elements:
    query = f"elements HAS ANY {el}"  # containing the el from refractory elements 
    #query = "elements HAS ANY " + ",".join(el)
    print("Scanning:", el)
    results = jarvis_optimade_query(query)
    entries.extend(results)  # extend statt append!

unique_entries = {e["id"]: e for e in entries}
entries = list(unique_entries.values())
print("Found number of unique materials:", len(entries))



# Filter for local properties

materials_data = []

for entry in entries:
    attr = entry["attributes"]
    
    # Basis-Daten
    mat_dict = {
        "material_id": entry["id"],
        "Chemical Formula": attr.get("chemical_formula_reduced"),
        "Chemical Formula Anonymus": attr.get("chemical_formula_anonymous"),
        "Chemical Formula Descriptive": attr.get("chemical_formula_descriptive"),
        "Bulk_modulus Voigt (GPa)": attr.get("_jarvis_bulk_modulus_kv"),
        "Shear modulus Voigt (GPa)": attr.get("_jarvis_shear_modulus_gv"),
        "Poisson ratio": attr.get("_jarvis_poisson"),
        "Crystal System": attr.get("_jarvis_crys"),
        "Spacegroup Symbol": attr.get("_jarvis_spg_symbol"),
        "Density (g/cm3)": attr.get("_jarvis_density"),
        "n_elements": attr.get("nelements"),
        "Material Type": attr.get("_jarvis_typ"),
        "Band Gap": attr.get("_jarvis_mbj_bandgap"),
        "Dimensionality": attr.get("_jarvis_dimensionality") 
    }

    # Calculation of lattice vectors and adding them to the dictionary 
    lattice_vectors = attr.get("lattice_vectors")
    if lattice_vectors:
        mat_dict.update(lattice_vectors_to_parameters(lattice_vectors))
    
    materials_data.append(mat_dict)

df = pd.DataFrame(materials_data)
print(df.shape) 
print(df.head())

base_folder = "Jarvis_Data"
os.makedirs(base_folder, exist_ok=True)
name = "Refractory_Metals_Data"
safe_name = re.sub(r'[<>:"/\\|?*]', "_", name)
csv_filepath = os.path.join(base_folder, f"{safe_name}.csv")
df.to_csv(csv_filepath, index=False)

print("Done. Dataframe contains all formulas and their properties.")